In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import re
import unicodedata

tanit = pd.read_csv("/kaggle/input/datasets/wissemweslati0/tanit-jobs1/tanitjobs_it_jobs (1).csv")

# 2. Normalize text (lowercase, remove accents)

def normalize_text(text):
    text = str(text)
    text = unicodedata.normalize("NFD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")
    return text.lower().strip()


# 3. Parse posting date

def parse_posting_date(text):
    if pd.isna(text):
        return None

    text = normalize_text(text)
    today = datetime.today()

    # Regex supports: il y a / il'y a
    pattern = r"il\s*'?y\s*a\s*(\d+)\s*(jour|jours|semaine|semaines|mois)"
    match = re.search(pattern, text)

    if not match:
        return None

    value = int(match.group(1))
    unit = match.group(2)

    if "jour" in unit:
        date = today - timedelta(days=value)
    elif "semaine" in unit:
        date = today - timedelta(weeks=value)
    elif "mois" in unit:
        date = today - timedelta(days=value * 30)  # approx

    return date.strftime("%d/%m/%y")


# 4. Apply transformation

tanit["posting_date"] = tanit["posting_date"].apply(parse_posting_date)


# 5. Display result

tanit

In [ ]:

# Load datasets
naukrigulf = pd.read_csv('/kaggle/input/datasets/wissemweslati0/keejob-naukrigulf/naukrigulf_selenium.csv')
keejob = pd.read_csv('/kaggle/input/datasets/wissemweslati0/keejob-naukrigulf/keejob_listings.csv')
linkedin = pd.read_csv('/kaggle/input/datasets/wissemweslati0/linkedin/linkedin_jobs_complete.csv')

# LinkedIn columns to keep
linkedin_cols = ['title', 'company', 'location', 'date', 'job_link', 'description', 'search_keyword']

# Map NaukriGulf columns to LinkedIn structure
naukrigulf_mapped = pd.DataFrame({
    'title': naukrigulf['title'],
    'company': naukrigulf['company'],
    'location': naukrigulf['location'],
    'date': naukrigulf['posted_date'],
    'job_link': naukrigulf['url'],
    'description': naukrigulf['full_description'],
    'search_keyword': None
})

# Map Keejob columns to LinkedIn structure
keejob_mapped = pd.DataFrame({
    'title': keejob['title'],
    'company': keejob['company'],
    'location': keejob['location'],
    'date': keejob['posted_date'],
    'job_link': keejob['url'],
    'description': keejob['full_description'],
    'search_keyword': None
})

tanit_mapped = pd.DataFrame({
    'title': tanit['job_title'],
    'company': tanit['company_name'],
    'location': tanit['location'],
    'date': tanit['posting_date'],
    'job_link': tanit['job_link'],
    'description': tanit['description'],
    'search_keyword': None
    
})

replacements = {
    "DonnÃ©es": "Données",
    "IngÃ©nieur": "Ingénieur",
    "DÃ©veloppeur": "Développeur",
    "SystÃ¨mes": "Systèmes",
    "confirmÃ©": "confirmé",
    "connectivitÃ©": "connectivité",
    "OpÃ©rateur": "Opérateur",
    "TÃ©lÃ©vendeur(Se)": "Télévendeur(Se)",
    "dâ€™entreprise": "d'entreprise"
}




# Keep only LinkedIn columns from the linkedin dataset
linkedin_mapped = linkedin[linkedin_cols]

# Merge all three
merged = pd.concat([linkedin_mapped, naukrigulf_mapped, keejob_mapped, tanit_mapped], ignore_index=True)

print(merged.shape)
print(merged.head())

merged.to_csv('merged_jobs.csv', index=False)

In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/working/merged_jobs.csv')

for col in linkedin_cols:
    for bad, good in replacements.items():
        df[col] = df[col].astype(str).str.replace(bad, good, regex=False)

print(f"Dataset loaded: {len(df)} jobs")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample job titles:")
print(df['title'].value_counts().head(20))





In [ ]:
def extract_relevant_section(text):
    """
    Extract only the relevant parts: missions, responsibilities, qualifications
    Skip company description
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Keywords that mark START of relevant content (French & English)
    start_keywords = [
        # French
        r'vos challenges',
        r'votre mission',
        r'description du poste',
        r'missions principales',
        r'vos missions',
        r'responsabilités',
        r'about the role',
        r'what you.?ll do',
        r'responsibilities',
        r'job description',
        r'your missions',
        r'key responsibilities',
        r'missions',
        r'poste',
        # English
        r'role overview',
        r'position summary',
        r'what you will do',
        r'duties',
    ]

    # Find the earliest match
    text_lower = text.lower()
    start_idx = len(text)  # Default to end if not found

    for keyword in start_keywords:
        match = re.search(keyword, text_lower)
        if match:
            start_idx = min(start_idx, match.start())

    # If no relevant section found, use full text (but this is rare)
    if start_idx == len(text):
        return text

    return text[start_idx:]


# Apply extraction
print("Extracting relevant sections from descriptions...")
df['relevant_section'] = df['description'].apply(extract_relevant_section)

# Compare before/after
print("\n" + "="*60)
print("EXAMPLE: Before extraction")
print("="*60)
print(df['description'].iloc[1][:400])

print("\n" + "="*60)
print("EXAMPLE: After extraction (relevant section only)")
print("="*60)
print(df['relevant_section'].iloc[1][:400])

print(f"\nAverage length before: {df['description'].str.len().mean():.0f} chars")
print(f"Average length after: {df['relevant_section'].str.len().mean():.0f} chars")


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download required NLTK data (run once)
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
except:
    pass

def preprocess_text(text):
    """Clean text for analysis"""
    if pd.isna(text) or text == "":
        return ""

    text = str(text).lower()

    # Remove URLs and emails
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # Remove punctuation but keep spaces
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove digits
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def remove_stopwords_bilingual(text):
    """Remove English and French stopwords"""
    if not text:
        return ""

    # Get stopwords for both languages
    stop_words = set()
    try:
        stop_words.update(stopwords.words('english'))
        stop_words.update(stopwords.words('french'))
    except:
        pass

    # Add custom stopwords (job-related generic terms)
    custom_stops = {
        'experience', 'expérience', 'ans', 'years', 'année', 'annees',
        'poste', 'job', 'work', 'travail', 'company', 'entreprise',
        'candidat', 'candidate', 'profil', 'profile', 'recherche',
        'looking', 'team', 'équipe', 'equipe', 'opportunity', 'opportunité',
        'role', 'position'
    }
    stop_words.update(custom_stops)

    words = text.split()
    filtered = [w for w in words if w not in stop_words and len(w) > 2]

    return " ".join(filtered)

In [ ]:
"""def lemmatize_text(text):
   
    if not text:
        return ""

    lemmatizer = WordNetLemmatizer()
    words = text.split()
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized)

# Apply preprocessing pipeline
print("Preprocessing text...")
df['text_cleaned'] = (
    df['relevant_section']
    .apply(preprocess_text)
    .apply(remove_stopwords_bilingual)
    .apply(lemmatize_text)
)

print("✓ Preprocessing complete")
print(f"Average words per description: {df['text_cleaned'].str.split().str.len().mean():.0f}")
"""


In [ ]:
# ============================================================
# IMPROVED JOB TITLE CATEGORIZATION
# Cell to replace Cell 4 in the original notebook
# ============================================================
"""
def categorize_job_title(title):

    if pd.isna(title):
        return 'Other', 'Other'

    title_lower = str(title).lower()

    # Remove common prefixes/suffixes to extract core role
    title_clean = title_lower
    for prefix in ['senior', 'junior', 'lead', 'principal', 'staff', 'confirmed', 'confirmé']:
        title_clean = title_clean.replace(prefix, '')
    title_clean = title_clean.strip()

    # ============================================================
    # DATA & ANALYTICS ROLES
    # ============================================================
    if any(word in title_lower for word in ['data scientist', 'scientist']):
        return 'Data & Analytics', 'Data Scientist'

    if any(word in title_lower for word in ['data engineer', 'ingénieur data', 'ingenieur data']):
        return 'Data & Analytics', 'Data Engineer'

    if any(word in title_lower for word in ['data analyst', 'analyste de données', 'analyste']):
        return 'Data & Analytics', 'Data Analyst'

    if any(word in title_lower for word in ['ml engineer', 'machine learning', 'mlops']):
        return 'Data & Analytics', 'ML Engineer'

    if 'bi' in title_clean or 'business intelligence' in title_lower:
        return 'Data & Analytics', 'BI Developer'

    if 'dataops' in title_lower:
        return 'Data & Analytics', 'DataOps Engineer'

    # ============================================================
    # AI & MACHINE LEARNING
    # ============================================================
    if any(word in title_lower for word in ['ai engineer', 'artificial intelligence', 'intelligence artificielle', 'gen ai', 'generative ai']):
        return 'AI & ML', 'AI Engineer'

    # ============================================================
    # SOFTWARE ENGINEERING
    # ============================================================
    if any(word in title_lower for word in ['software engineer', 'software developer', 'ingénieur logiciel']):
        # Check for specialization
        if 'embedded' in title_lower:
            return 'Software Engineering', 'Embedded Engineer'
        if 'qa' in title_lower or 'quality' in title_lower:
            return 'Software Engineering', 'QA Engineer'
        return 'Software Engineering', 'Software Engineer'

    # Backend
    if any(word in title_lower for word in ['backend', 'back-end', 'back end']):
        return 'Software Engineering', 'Backend Developer'

    # Frontend
    if any(word in title_lower for word in ['frontend', 'front-end', 'front end']):
        return 'Software Engineering', 'Frontend Developer'

    # Full Stack
    if any(word in title_lower for word in ['full stack', 'fullstack', 'full-stack']):
        return 'Software Engineering', 'Full Stack Developer'

    # Technology-specific developers
    if '.net' in title_lower or 'dotnet' in title_lower:
        return 'Software Engineering', '.NET Developer'

    if 'java' in title_lower and 'javascript' not in title_lower:
        return 'Software Engineering', 'Java Developer'

    if 'python' in title_lower:
        return 'Software Engineering', 'Python Developer'

    if any(word in title_lower for word in ['angular', 'react', 'vue']):
        return 'Software Engineering', 'Frontend Developer'

    if 'golang' in title_lower or 'go developer' in title_lower:
        return 'Software Engineering', 'Go Developer'

    # Generic developer
    if any(word in title_lower for word in ['developer', 'développeur', 'developpeur']):
        return 'Software Engineering', 'Developer'

    # ============================================================
    # DEVOPS & CLOUD
    # ============================================================
    if any(word in title_lower for word in ['devops', 'dev ops']):
        return 'DevOps & Cloud', 'DevOps Engineer'

    if any(word in title_lower for word in ['cloud engineer', 'cloud architect', 'aws', 'azure']):
        return 'DevOps & Cloud', 'Cloud Engineer'

    if 'sre' in title_lower or 'site reliability' in title_lower:
        return 'DevOps & Cloud', 'SRE'

    # ============================================================
    # QUALITY ASSURANCE & TESTING
    # ============================================================
    if any(word in title_lower for word in ['qa', 'quality assurance', 'test engineer', 'tester', 'automation test']):
        return 'Quality & Testing', 'QA Engineer'

    # ============================================================
    # SECURITY
    # ============================================================
    if any(word in title_lower for word in ['security', 'cybersecurity', 'cyber', 'secops', 'ciso']):
        return 'Security', 'Security Engineer'

    if 'soc analyst' in title_lower:
        return 'Security', 'SOC Analyst'

    # ============================================================
    # DESIGN & UX
    # ============================================================
    if any(word in title_lower for word in ['ux', 'ui', 'ux/ui', 'ui/ux']):
        return 'Design', 'UI/UX Designer'

    if any(word in title_lower for word in ['graphic designer', 'graphiste', 'motion designer', 'motion graphic']):
        return 'Design', 'Graphic Designer'

    if 'designer' in title_lower:
        return 'Design', 'Designer'

    # ============================================================
    # SUPPORT & INFRASTRUCTURE
    # ============================================================
    if any(word in title_lower for word in ['support', 'helpdesk', 'service desk', 'technicien it']):
        return 'IT Support', 'IT Support'

    if any(word in title_lower for word in ['network', 'infrastructure', 'system admin', 'administrateur']):
        return 'IT Support', 'System/Network Admin'

    if 'dba' in title_lower or 'database admin' in title_lower:
        return 'IT Support', 'Database Admin'

    # ============================================================
    # CONSULTING & PROJECT MANAGEMENT
    # ============================================================
    if 'consultant' in title_lower:
        if 'sap' in title_lower:
            return 'Consulting', 'SAP Consultant'
        if 'crm' in title_lower:
            return 'Consulting', 'CRM Consultant'
        if 'functional' in title_lower:
            return 'Consulting', 'Functional Consultant'
        return 'Consulting', 'IT Consultant'

    if any(word in title_lower for word in ['project manager', 'chef de projet', 'pmo']):
        return 'Project Management', 'Project Manager'

    # ============================================================
    # BUSINESS & OPERATIONS
    # ============================================================
    if any(word in title_lower for word in ['business analyst', 'analyste business']):
        return 'Business & Operations', 'Business Analyst'

    if any(word in title_lower for word in ['product manager', 'product owner']):
        return 'Business & Operations', 'Product Manager'

    if any(word in title_lower for word in ['business developer', 'business development']):
        return 'Business & Operations', 'Business Developer'

    # ============================================================
    # INTERNSHIPS
    # ============================================================
    if any(word in title_lower for word in ['intern', 'internship', 'stagiaire', 'stage', 'pfe']):
        return 'Internship', 'Internship'

    # ============================================================
    # CATCH-ALL FOR ENGINEERING
    # ============================================================
    if 'engineer' in title_lower or 'ingénieur' in title_lower or 'ingenieur' in title_lower:
        return 'Engineering', 'Engineer'

    # ============================================================
    # OTHER
    # ============================================================
    return 'Other', 'Other'


# Apply the categorization
df['job_category'], df['job_subcategory'] = zip(*df['title'].apply(categorize_job_title))

# Show distribution
print("="*70)
print("JOB CATEGORIES DISTRIBUTION")
print("="*70)
category_dist = df['job_category'].value_counts()
print(category_dist)

print(f"\n{'='*70}")
print(f"'Other' category: {category_dist.get('Other', 0)} jobs ({category_dist.get('Other', 0)/len(df)*100:.1f}%)")
print(f"{'='*70}")

# Show subcategories
print(f"\n{'='*70}")
print("TOP 20 SUBCATEGORIES")
print(f"{'='*70}")
subcategory_dist = df['job_subcategory'].value_counts()
print(subcategory_dist.head(20))

# Show what's in "Other"
if 'Other' in category_dist.index:
    print(f"\n{'='*70}")
    print("JOBS CATEGORIZED AS 'OTHER' (sample):")
    print(f"{'='*70}")
    other_titles = df[df['job_category'] == 'Other']['title'].value_counts().head(15)
    for title, count in other_titles.items():
        print(f"{count:3d} - {title}")

# Visualize main categories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Main categories
category_dist.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Number of Jobs', fontsize=12)
ax1.set_title('Job Categories Distribution', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# Top subcategories
subcategory_dist.head(15).plot(kind='barh', ax=ax2, color='coral')
ax2.set_xlabel('Number of Jobs', fontsize=12)
ax2.set_title('Top 15 Job Subcategories', fontsize=14, fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('job_categories_improved.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Categorization complete!")
print(f"✓ Using 'job_category' for broad grouping")
print(f"✓ Using 'job_subcategory' for detailed analysis")


# ============================================================
# OPTIONAL: Use subcategories for more granular skill extraction
# ============================================================

print(f"\n{'='*70}")
print("RECOMMENDATION FOR SKILL EXTRACTION:")
print(f"{'='*70}")

# Show which subcategories have enough data for reliable extraction
subcats_with_enough_data = subcategory_dist[subcategory_dist >= 2]
print(f"\nSubcategories with 5+ jobs (good for skill extraction):")
print(f"Total: {len(subcats_with_enough_data)} subcategories")
print(subcats_with_enough_data)
"""

In [ ]:
"""df['job_category'] = df['title'].apply(normalize_job_title)

# Show distribution
print("Job categories distribution:")
print(df['job_category'].value_counts())

# Visualize
plt.figure(figsize=(12, 6))
df['job_category'].value_counts().head(15).plot(kind='barh')
plt.xlabel('Number of Jobs')
plt.title('Top Job Categories in Dataset')
plt.tight_layout()
plt.savefig('job_categories_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
"""

In [ ]:
print(f"Total jobs: {len(df)}")
print(f"Unique job titles: {df['title'].nunique()}")


# ============================================================
# STEP 1: Clean and normalize job titles
# ============================================================

def clean_title(title):
    """Basic cleaning without categorization"""
    if pd.isna(title):
        return ""

    title = str(title).lower()

    # Remove common noise
    noise_terms = [
        r'\(h/f\)', r'\(f/m/x\)', r'\(m/f/x\)', r'\(w/m\)',
        r'\[f/m/x\]', r'h/f', r'f/h', r'm/f', r'w/m',
        r'- tunis', r'tunis -', r'remote', r'relocat.*',
        r'located geographically.*', r'\(.*portugal.*\)',
        r'\(.*dubai.*\)', r'internship', r'stagiaire'
    ]

    import re
    for pattern in noise_terms:
        title = re.sub(pattern, '', title, flags=re.IGNORECASE)

    # Remove extra spaces
    title = re.sub(r'\s+', ' ', title).strip()

    return title

df['title_cleaned'] = df['title'].apply(clean_title)

print(f"\nCleaned titles - sample:")
print(df[['title', 'title_cleaned']].head(10))



In [ ]:
unique_titles = df['title_cleaned'].unique()
unique_titles = [t for t in unique_titles if len(t) > 2]  # Remove very short titles

print(f"\n{len(unique_titles)} unique cleaned titles")

from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectors
vectorizer = TfidfVectorizer(
    analyzer='char_wb',  # Character n-grams to handle typos
    ngram_range=(2, 4),  # 2-4 character sequences
    min_df=1,
    max_df=0.8
)

title_vectors = vectorizer.fit_transform(unique_titles)

print(f"TF-IDF matrix shape: {title_vectors.shape}")

In [ ]:
# Compute similarity matrix
similarity_matrix = cosine_similarity(title_vectors)

# Convert similarity to distance
distance_matrix = 1 - similarity_matrix

# Cluster using Agglomerative Clustering
# n_clusters: adjust based on how many categories you want
# Higher = more specific categories, Lower = broader categories

n_clusters = 30  # Start with 30, adjust as needed

clustering = AgglomerativeClustering(
    n_clusters=n_clusters,
    metric='precomputed',
    linkage='average'
)

cluster_labels = clustering.fit_predict(distance_matrix)

# Create mapping from title to cluster
title_to_cluster = dict(zip(unique_titles, cluster_labels))

print(f"\nClustered {len(unique_titles)} titles into {n_clusters} groups")


In [ ]:
from collections import Counter
import re

def extract_keywords(titles):
    """Extract most common keywords from a list of titles"""
    all_words = []

    for title in titles:
        # Remove common words
        stop_words = ['senior', 'junior', 'lead', 'principal', 'confirmé',
                     'confirmed', 'intern', 'stagiaire', 'stage']
        words = title.split()
        words = [w for w in words if w not in stop_words and len(w) > 2]
        all_words.extend(words)

    # Get most common
    word_counts = Counter(all_words)

    if not word_counts:
        return "Uncategorized"

    # Take top 2 most common words
    top_words = [word for word, count in word_counts.most_common(2)]
    return " ".join(top_words).title()


In [ ]:
# Group titles by cluster and name each cluster
cluster_names = {}
cluster_titles = {}

for cluster_id in range(n_clusters):
    # Get all titles in this cluster
    titles_in_cluster = [title for title, cid in title_to_cluster.items() if cid == cluster_id]
    cluster_titles[cluster_id] = titles_in_cluster

    # Generate name
    cluster_name = extract_keywords(titles_in_cluster)
    cluster_names[cluster_id] = cluster_name

print(f"\n{'='*70}")
print("AUTOMATICALLY GENERATED CATEGORIES")
print(f"{'='*70}")

for cluster_id in sorted(cluster_names.keys()):
    titles = cluster_titles[cluster_id]
    print(f"\n{cluster_names[cluster_id]} ({len(titles)} titles):")
    print("  Sample titles:", ", ".join(titles[:5]))


In [ ]:
# Map each job to its cluster
df['cluster_id'] = df['title_cleaned'].map(title_to_cluster)
df['auto_category'] = df['cluster_id'].map(cluster_names)

# Handle any unmapped titles
df['auto_category'] = df['auto_category'].fillna('Other')

# Show distribution
print(f"\n{'='*70}")
print("CATEGORY DISTRIBUTION")
print(f"{'='*70}")
category_dist = df['auto_category'].value_counts()
print(category_dist)

print(f"\n'Other' category: {category_dist.get('Other', 0)} jobs ({category_dist.get('Other', 0)/len(df)*100:.1f}%)")


In [ ]:
category_refinement = {
    'Manager Product': 'Product & Project Management',
    'Officer Directeur': 'Management & Leadership',
    'Team Tech': 'Tech Leadership',
    'Chef Projet': 'Project Management',

    # IT Support & Administration
    'Administrator Administrateur': 'System Administration',
    'Support Specialist': 'IT Support & Technical Services',

    # Software Engineering & Development
    'Developer Full': 'Software Engineering',
    'Engineer Software': 'Software Engineering',
    'Ingénieur Devops': 'DevOps Engineering',

    # Data & Analytics
    'Data Scientist': 'Data Science & ML',
    'Analyst Business': 'Data & Business Analysis',

    # Design & Creative
    'Video Editor': 'Multimedia & Video Production',
    'Designer Graphic': 'Design & Graphics',

    # Architecture & Infrastructure
    'Architect Architecte': 'Solution Architecture',

    # Consulting
    'Consultant Sap': 'IT Consulting & ERP',

    # Business & Operations
    'Communication Application': 'Business Operations',
    'Assistant Executive': 'Administrative Support',
    'Recruiter (Freelance': 'Recruitment',
    'Buyer Procurement': 'Procurement & Supply Chain',
    'Associate Rims': 'Operations',

    # Specialized Roles
    'Maximo Developer': 'Enterprise Software Development',
    'Pharmacist': 'Healthcare & Quality',
    'Functional Function': 'Functional Engineering',
    'Webmaster': 'Web Administration',
    'Magasinier': 'Warehouse & Logistics',
    'Safran Electrical': 'Aerospace & Industrial Engineering',
    'Opportunity Build': 'Special Projects & Internships',
    'Placier (Zones': 'Field Operations',

    # Finance & Control
    'Contrôle Controller': 'Finance & Control',

    # Keep as Other (not relevant or too vague)
    'Mental Health': 'Other',
}

# Apply refinement
df['final_category'] = df['auto_category'].map(
    lambda x: category_refinement.get(x, x)
)
category_dist = df['final_category'].value_counts()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 categories
top_categories = category_dist.head(15)
top_categories.plot(kind='barh', ax=ax1, color='teal')
ax1.set_xlabel('Number of Jobs', fontsize=12)
ax1.set_title('Top 15 Automatically Detected Categories', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# Category sizes distribution
category_sizes = df['final_category'].value_counts().values
ax2.hist(category_sizes, bins=20, color='coral', edgecolor='black')
ax2.set_xlabel('Jobs per Category', fontsize=12)
ax2.set_ylabel('Number of Categories', fontsize=12)
ax2.set_title('Distribution of Category Sizes', fontsize=14, fontweight='bold')
ax2.axvline(1, color='red', linestyle='--', label='Min threshold (5 jobs)')
ax2.legend()

plt.tight_layout()
plt.savefig('auto_clustering_results.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
valid_categories = category_dist[category_dist >= 5].index.tolist()

print(f"\n{'='*70}")
print(f"CATEGORIES WITH 5+ JOBS (good for skill extraction)")
print(f"{'='*70}")
print(f"Total: {len(valid_categories)} categories")
print(f"\n{category_dist[category_dist >= 5]}")


In [ ]:
mapping_df = pd.DataFrame([
    {
        'cluster_id': cid,
        'category_name': cluster_names[cid],
        'num_titles': len(cluster_titles[cid]),
        'sample_titles': ' | '.join(cluster_titles[cid][:3])
    }
    for cid in sorted(cluster_names.keys())
])

mapping_df.to_csv('cluster_mapping.csv', index=False)
print(f"\n✓ Saved cluster mapping to 'cluster_mapping.csv'")

# Save enhanced dataframe
df.to_csv('linkedin_jobs_with_auto_categories.csv', index=False)
print(f"✓ Saved dataset with auto categories to 'linkedin_jobs_with_auto_categories.csv'")



In [ ]:
df.info()

In [ ]:
df